# Lektion 01 - Einführung in KI-Agenten

Willkommen zur ersten Lektion im Kurs **KI-Agenten für Anfänger**!

Ein **KI-Agent** ist ein Programm, das ein großes Sprachmodell (LLM) als seine Denkmaschine verwendet und *Aktionen* in der realen Welt ausführen kann – APIs aufrufen, Datenbanken abfragen oder Code ausführen – um im Auftrag eines Nutzers ein Ziel zu erreichen.

In diesem Notebook baust du deinen ersten Agenten: einen **Reiseagenten**, der Urlaubsdestinationen empfiehlt. Dabei wirst du lernen, wie man:

1. Eine Verbindung zum Microsoft Foundry Agent Service über das **Microsoft Agent Framework** herstellt.
2. Dem Agenten ein **Tool** gibt – eine einfache Python-Funktion, die er aufrufen kann.
3. Den Agenten ausführt und seine Antwort überprüft.
4. Die Antwort des Agenten Token für Token streamt.


## Einrichtung

Bevor Sie dieses Notebook ausführen, stellen Sie sicher, dass Sie:

1. **Ein Microsoft Foundry-Projekt** mit einem bereitgestellten Chatmodell (z. B. `gpt-5-mini`) haben.
2. **Mit der Azure CLI angemeldet sind** — führen Sie `az login` in Ihrem Terminal aus.
3. **Die erforderlichen Umgebungsvariablen gesetzt haben:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Ihr Microsoft Foundry-Projekt-Endpunkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — der Name Ihres bereitgestellten Modells.

Die folgende Zelle installiert die benötigten Python-Pakete.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Erstellen Ihres ersten Agenten

Ein Agent benötigt zwei Dinge:

- **Anweisungen**, die ihm *sagen, wer er ist* und *wie er sich verhalten soll* (ein System-Prompt).
- **Werkzeuge** — Python-Funktionen, die mit `@tool` dekoriert sind und die der Agent aufrufen kann, um Informationen abzurufen oder Aktionen auszuführen.

Unten definieren wir ein einfaches Werkzeug, das eine Liste beliebter Urlaubsziele zurückgibt. Der Agent wird dieses Werkzeug verwenden, wenn ein Benutzer nach Reiseempfehlungen fragt.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming-Antworten

Für ein interaktiveres Erlebnis können Sie die Antwort des Agenten **streamen**. Statt auf die vollständige Antwort zu warten, liefert der Agent Textabschnitte, sobald sie generiert werden. Dies ist besonders nützlich in Chat-Oberflächen, in denen Sie die Ausgabe in Echtzeit anzeigen möchten.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Zusammenfassung

In dieser Lektion hast du gelernt, wie man:

- **Einen Provider erstellt**, der sich über `FoundryChatClient` mit dem Microsoft Foundry Agent Service verbindet.
- **Ein Tool definiert** mit dem `@tool` Dekorator, damit der Agent deine Python-Funktionen aufrufen kann.
- **Den Agenten startet** mit einer Benutzernachricht und seine Antwort ausgibt.
- **Antworten streamt** für eine Echtzeitausgabe.

In der nächsten Lektion werden wir agentenbasierte Frameworks genauer untersuchen und lernen, wie man Agenten leistungsfähigere Werkzeuge und mehrstufige Denkfähigkeiten gibt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
